In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
schema_path='s3://john-deere-ro/schema-path'
raw_source_path='s3://john-deere-ro/raw_incentive_sales/'
checkpoint_path='s3://john-deere-ro/checkpoint-path'

raw incentive data bronze

In [0]:
raw_incentives_sales_bronze=spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','json')\
        .option('cloudFiles.schemaLocation',schema_path)\
            .option('cloudFiles.inferColumnTypes','true')\
                .option('cloudFiles.rescuedDataColumn','_res_data')\
                    .load(raw_source_path)
raw_incentives_sales_bronze.writeStream.format('delta')\
    .option('checkpointLocation',checkpoint_path)\
        .option('mergeSchema','true')\
            .trigger(availableNow=True)\
                .outputMode('Append')\
                    .table('johndeere.default.raw_incentives_sales_bronze')

dim dealers raw

In [0]:

dim_dealers_raw_bronze = (
    spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format','parquet')
        .option('cloudFiles.schemaLocation', schema_path)\
          .option('cloudFiles.schemaEvolutionMode','addNewColumns')

        .option('cloudFiles.rescuedDataColumn','_res_data')
        .option('cloudFiles.inferColumnTypes','true')
        .load('s3://john-deere-ro/dim_dealers_raw_folder/')
)

dim_dealers_raw_bronze.writeStream.format('delta')\
  .option('checkpointLocation','s3://john-deere-ro/dim-dealers-checkpoint-path')\
    .outputMode('Append')\
      .trigger(availableNow=True)\
        .option('mergeSchema','true')\
          .table('johndeere.default.dim_dealers_bronze_table')

dim equipments raw

In [0]:
dim_equipments_bronze=(
    spark.readStream.format('cloudFiles')\
        .option('cloudFiles.format','parquet')\
            .option('cloudFiles.schemaLocation','s3://john-deere-ro/dim-equip-schema-path')\
                .option('cloudFiles.schemaEvolutionMode','addNewColumns')\
                    .option('cloudFiles.inferColumnTypes','true')\
                        .option('cloudFiles.rescuedDataColumn','_res_data')\
                            .load('s3://john-deere-ro/dim_equipment_raw/')

)
dim_equipments_bronze.writeStream.format('delta')\
    .option('checkpointLocation','s3://john-deere-ro/dim-equipments-checkpoint-path')\
        .outputMode('Append')\
            .trigger(availableNow=True)\
                .option('mergeSchema','true')\
                    .table('johndeere.default.dim_equipments_bronze_table')

In [0]:
legacy_payout_table=(
    spark.readStream.format('cloudFiles')\
        .option('cloudFiles.format','parquet')\
            .option('cloudFiles.schemaLocation','s3://john-deere-ro/legacy-schema-path')\
                .option('cloudFiles.schemaEvolutionMode','addNewColumns')\
                    .option('cloudFiles.inferColumnTypes','true')\
                        .option('cloudFiles.rescuedDataColumn','_res_data')\
                            .load('s3://john-deere-ro/legacy_payouts_mysql/')

)
legacy_payout_table.writeStream.format('delta')\
    .option('checkpointLocation','s3://john-deere-ro/legacy-checkpoint-path')\
        .outputMode('Append')\
            .trigger(availableNow=True)\
                .option('mergeSchema','true')\
                    .table('johndeere.default.legacy_payout_bronze_table')